# Chapter 10
 Machine Learning for Business Analytics<br>
Concepts, Techniques, and Applications in Python<br>
by Galit Shmueli, Peter C. Bruce, Peter Gedeck, Nitin R. Patel

Publisher: Wiley; 2nd edition (2024) <br>
<!-- ISBN-13: 978-3031075650 -->

(c) 2024 Galit Shmueli, Peter C. Bruce, Peter Gedeck, Nitin R. Patel

The code needs to be executed in sequence.

Python packages and Python itself change over time. This can cause warnings or errors.
"Warnings" are for information only and can usually be ignored.
"Errors" will stop execution and need to be fixed in order to get results.

If you come across an issue with the code, please follow these steps

- Check the repository (https://gedeck.github.io/sdsa-code-solutions/) to see if the code has been upgraded. This might solve the problem.
- Report the problem using the issue tracker at https://github.com/gedeck/sdsa-code-solutions/issues
- Paste the error message into Google and see if someone else already found a solution

**Cell 1: Import libraries**  
This cell imports necessary libraries: `matplotlib.pyplot` for plotting, `mlba` for data loading and utility functions, `numpy` and `pandas` for data manipulation, `seaborn` for advanced visualisations, `statsmodels.api` for statistical modeling, `mord.LogisticIT` for ordinal logistic regression, and `sklearn.linear_model` for logistic regression and cross‑validated logistic regression. The `%matplotlib inline` ensures plots are displayed inline.

In [ ]:
import matplotlib.pyplot as plt
import mlba
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from mord import LogisticIT
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split
%matplotlib inline

**Cell 2: Visualise the logit (log‑odds) transformation**  
This cell creates a grid of plots showing the relationship between probability of success (p), odds (p/(1‑p)), and logit (log odds). The first subplot shows odds increasing non‑linearly from 0 to 100 as p goes from 0 to 1. The second subplot shows the logit, which is linear in the log‑odds scale, ranging from -∞ to +∞. These plots illustrate why logistic regression models the logit as a linear function of predictors – it maps the bounded probability (0,1) to an unbounded continuous scale.

In [ ]:
p = np.arange(0.001, 0.999, 0.001)
df = pd.DataFrame({
    'Probability of success': p,
    'Odds': p / (1 - p),
    'Logit': np.log(p / (1 - p)),
})
fig, axes = plt.subplots(nrows = 2, figsize = (6, 6))
df.plot(x = 'Probability of success', y = 'Odds', ax = axes[0], ylim = (0, 100))
axes[0].set_ylabel('Odds')
axes[1].axhline(y = 0, color = 'black', linestyle = '-')
df.plot(x = 'Probability of success', y = 'Logit', ax = axes[1], ylim = (-4, 4))
axes[1].set_ylabel('Logit')
for ax in axes:
    ax.legend().set_visible(False)
plt.tight_layout()
plt.show()

**Cell 3: Logistic regression curve for a single predictor (Income) on Universal Bank data**  
The Universal Bank dataset is loaded. A logistic regression model with only `Income` as predictor is approximated using coefficients from a previously fitted model (intercept ≈ 6.05, coefficient ≈ -0.036). The scatter plot shows the binary `Personal Loan` outcome (jittered with alpha transparency) and the fitted sigmoid curve. The curve shows that as income increases, the probability of accepting a personal loan increases, which makes intuitive sense (higher income individuals are more likely to be offered and accept loans).

In [ ]:
bank_df = mlba.load_data('UniversalBank.csv')

income = np.arange(0, 250, 1)
df = pd.DataFrame({
    'Income': income,
    'p': 1 / (1 + np.exp(6.04892 - 0.036 * income)),
})

ax = bank_df.plot.scatter(x='Income', y='Personal Loan', alpha=0.1)
ax.plot(df['Income'], df['p'], color='black')
ax.set_ylabel('Personal loan')
plt.tight_layout()
plt.show()

**Cell 4: Fit a full logistic regression on Universal Bank data (all predictors)**  
The Universal Bank dataset is prepared: `ID` and `ZIP Code` are dropped; `Education` is converted to a categorical variable with meaningful labels and then one‑hot encoded (dropping the first category to avoid multicollinearity). The data is split into training (60%) and holdout (40%) sets. A logistic regression model with no regularisation (C=1e42, penalty='l2') is fitted. The intercept and coefficients are printed, followed by the AIC score (computed on the holdout set). The coefficients indicate the direction and magnitude of each predictor’s effect on the log‑odds of accepting a loan.

In [ ]:
bank_df = mlba.load_data('UniversalBank.csv')
bank_df = bank_df.drop(columns=['ID', 'ZIP Code'])

# Treat education as categorical, convert to dummy variables
bank_df['Education'] = bank_df['Education'].astype('category')
new_categories = {1: 'Undergrad', 2: 'Graduate', 3: 'Advanced/Professional'}
bank_df['Education'] = bank_df.Education.cat.rename_categories(new_categories)
bank_df = pd.get_dummies(bank_df, prefix_sep='_', drop_first=True, dtype=int)

y = bank_df['Personal Loan']
X = bank_df.drop(columns=['Personal Loan'])

# partition data
train_X, holdout_X, train_y, holdout_y = train_test_split(X, y, test_size=0.4, random_state=1)

# fit a logistic regression (set penalty=l2 and C=1e42 to avoid regularization)
logit_reg = LogisticRegression(penalty="l2", C=1e42, solver='liblinear')
logit_reg.fit(train_X, train_y)

print(f'Intercept {logit_reg.intercept_[0]:.6f}')
print(pd.DataFrame({'Coeff': logit_reg.coef_[0]}, index=X.columns).transpose())

aic_score = mlba.AIC_score(y_true=holdout_y, y_pred=logit_reg.predict(holdout_X),
                           df=len(train_X.columns)+1)
print(f'AIC {aic_score:.3f}')

**Cell 5: Predictions on holdout set for selected cases**  
The model is used to predict probabilities for the holdout set. Four specific rows (indices 2764, 932, 2721, 702) are displayed. For each, the table shows the actual loan status, predicted probability of class 0 (no loan) and class 1 (loan), and the predicted class (based on a 0.5 cutoff). This helps verify that the model is working as expected and to inspect borderline cases.

In [ ]:
logit_reg_proba = logit_reg.predict_proba(holdout_X)
logit_result = pd.DataFrame({'actual': holdout_y,
                             'p(0)': [p[0] for p in logit_reg_proba],
                             'p(1)': [p[1] for p in logit_reg_proba],
                             'predicted': logit_reg.predict(holdout_X) })

# display four different cases
interestingCases = [2764, 932, 2721, 702]
print(logit_result.loc[interestingCases])

**Cell 6: Gains and lift charts for the logistic model**  
Using the predicted probabilities of the positive class (loan acceptance), a gains chart (left) and a lift chart (right) are plotted. The gains chart shows the cumulative percentage of actual positives captured as we descend the ranked list. The lift chart shows the ratio of the cumulative positive rate to the overall positive rate for each decile. These plots help evaluate the model’s ranking performance – how well it concentrates the loan acceptors at the top of the list.

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
mlba.gainsChart(logit_result, ranking='p(1)', actual='actual', ax=axes[0])
mlba.liftChart(logit_result, ranking='p(1)', actual='actual', ax=axes[1])
plt.tight_layout()
plt.show()

**Cell 7: Nominal vs. ordinal logistic regression on accident severity**  
The `accidentsFull` dataset is used to model the ordinal outcome `MAX_SEV_IR` (severity levels 0, 1, 2) with two predictors (`ALCHL_I` and `WEATHER_R`). First, a nominal (multinomial) logistic regression is fitted using `LogisticRegression` with multinomial setting (though the code uses default `ovr`, but effectively it fits separate binary models). Coefficients and predicted probabilities are printed. Then, an ordinal logistic regression is fitted using `mord.LogisticIT` (which assumes proportional odds). The thresholds (`theta`) and coefficients are shown, along with predicted probabilities. Ordinal regression respects the ordering of the outcome and typically uses fewer parameters, which can be more parsimonious.

In [ ]:
data = mlba.load_data('accidentsFull.csv')
outcome = 'MAX_SEV_IR'
predictors = ['ALCHL_I', 'WEATHER_R']

y = data[outcome]
X = data[predictors]
train_X, train_y = X, y

print('Nominal logistic regression')
logit = LogisticRegression(penalty="l2", solver='lbfgs', C=1e24)
logit.fit(X, y)
print(f'  Intercept {logit.intercept_}')
print(f'  Coefficients\n{logit.coef_}')
print()
probs = logit.predict_proba(X)
results = pd.DataFrame({
    'actual': y, 'predicted': logit.predict(X),
    'P(0)': [p[0] for p in probs],
    'P(1)': [p[1] for p in probs],
    'P(2)': [p[2] for p in probs],
})
print(results.head().round(3))
print()

print('Ordinal logistic regression')
logit = LogisticIT(alpha=0)
logit.fit(X, y)
print(f'  Theta {logit.theta_}')
print(f'  Coefficients {logit.coef_}')
print()
probs = logit.predict_proba(X)
results = pd.DataFrame({
    'actual': y, 'predicted': logit.predict(X),
    'P(0)': [p[0] for p in probs],
    'P(1)': [p[1] for p in probs],
    'P(2)': [p[2] for p in probs],
})
print(results.head().round(3))

**Cell 8: Exploratory plots for flight delays – average delay by categorical predictors**  
The `FlightDelays` dataset is loaded, and a binary indicator `isDelayed` is created (1 if delayed, 0 otherwise). Several bar charts are created showing the average delay (proportion of delayed flights) for each level of: day of week, destination airport, departure hour (grouped), carrier, origin airport, and weather condition. These plots help identify which categories are associated with higher delay rates, providing guidance for feature engineering and model building.

In [ ]:
delays_df = mlba.load_data('FlightDelays.csv')
# Create an indicator variable
delays_df['isDelayed'] = [1 if status == 'delayed' else 0
                          for status in delays_df['Flight Status']]

def createGraph(group, xlabel, axis):
    groupAverage = delays_df.groupby([group])['isDelayed'].mean()
    if group == 'DAY_WEEK': # rotate so that display starts on Sunday
        groupAverage = groupAverage.reindex(index=np.roll(groupAverage.index,1))
        groupAverage.index = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
    ax = groupAverage.plot.bar(color='C0', ax=axis)
    ax.set_ylabel('Average delay')
    ax.set_xlabel(xlabel)
    return ax

def graphDepartureTime(xlabel, axis):
    temp_df = pd.DataFrame({'CRS_DEP_TIME': delays_df['CRS_DEP_TIME'] // 100,
                            'isDelayed': delays_df['isDelayed']})
    groupAverage = temp_df.groupby(['CRS_DEP_TIME'])['isDelayed'].mean()
    ax = groupAverage.plot.bar(color='C0', ax=axis)
    ax.set_xlabel(xlabel); ax.set_ylabel('Average delay')

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(10, 9))

createGraph('DAY_WEEK', 'Day of week', axis=axes[0][0])
createGraph('DEST', 'Destination', axis=axes[0][1])
graphDepartureTime('Departure time', axis=axes[1][0])
createGraph('CARRIER', 'Carrier', axis=axes[1][1])
createGraph('ORIGIN', 'Origin', axis=axes[2][0])
createGraph('Weather', 'Weather', axis=axes[2][1])
plt.tight_layout()
plt.show()


We introduced the column `isDelayed` to be able to easily compute the average delay for each group.

**Cell 9: Heatmaps of delay rates by origin, day of week, and carrier**  
For each origin airport (DCA, IAD, BWI), a heatmap is created showing the average delay rate for each combination of carrier and day of week. The colour intensity indicates the proportion of delayed flights. This visualisation helps identify problematic routes (e.g., specific carrier‑day‑origin combinations with high delay rates). A common colour bar is shared across the three heatmaps for consistent comparison.

In [ ]:
agg = delays_df.groupby(['ORIGIN', 'DAY_WEEK', 'CARRIER']).isDelayed.mean()
agg = agg.reset_index()

# Define the layout of the graph
height_ratios = []
for origin in sorted(delays_df.ORIGIN.unique()):
    height_ratios.append(len(agg[agg.ORIGIN == origin].CARRIER.unique()))
gridspec_kw = {'height_ratios': height_ratios, 'width_ratios': [15, 1]}
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(10, 6),
                         gridspec_kw = gridspec_kw)
axes[0, 1].axis('off')
axes[2, 1].axis('off')

maxIsDelay = agg.isDelayed.max()
for i, origin in enumerate(sorted(delays_df.ORIGIN.unique())):
    data = pd.pivot_table(agg[agg.ORIGIN == origin], values='isDelayed', aggfunc='sum',
                          index=['CARRIER'], columns=['DAY_WEEK'])
    data = data[[7, 1, 2, 3, 4, 5, 6]]  # Shift last columns to first
    ax = sns.heatmap(data, ax=axes[i][0], vmin=0, vmax=maxIsDelay,
                     cbar_ax=axes[1][1], cmap=sns.light_palette("navy"))
    ax.set_xticklabels(['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat'])
    if i != 2:
        ax.get_xaxis().set_visible(False)
    ax.set_xlabel('Day of week')
    ax.set_ylabel('Airport ' + origin)
plt.tight_layout()
plt.show()

**Cell 10: Full logistic regression model for flight delays (all categorical predictors)**  
The flight delays data is prepared: departure time is binned into hourly categories; all categorical predictors (`DAY_WEEK`, `CRS_DEP_TIME`, `ORIGIN`, `DEST`, `CARRIER`, `Weather`) are converted to dummy variables (dropping one category each). The outcome `Flight Status` is renamed so that the event of interest (‘delayed’) becomes the second category (ensuring correct probability estimation). The data is split into training and holdout sets. A logistic regression without regularisation is fitted, and the coefficients are printed (partial output). The AIC on the holdout set is computed. This model uses many dummy variables and may be overparameterised; later cells will explore reduced models.

In [ ]:
delays_df = mlba.load_data('FlightDelays.csv')

# create hourly bins departure time
delays_df['CRS_DEP_TIME'] = [round(t / 100) for t in delays_df['CRS_DEP_TIME']]

# convert to categorical
delays_df['DAY_WEEK'] = delays_df['DAY_WEEK'].astype('category')
delays_df['CRS_DEP_TIME'] = delays_df['CRS_DEP_TIME'].astype('category')
delays_df['Flight Status'] = delays_df['Flight Status'].astype('category')
new_categories = {'ontime': '0-ontime', 'delayed': '1-delayed'}
delays_df['Flight Status'] = delays_df['Flight Status'].cat.rename_categories(new_categories)

predictors = ['DAY_WEEK', 'CRS_DEP_TIME', 'ORIGIN', 'DEST', 'CARRIER', 'Weather']
outcome = 'Flight Status'

X = pd.get_dummies(delays_df[predictors], drop_first=True)
y = delays_df[outcome]

# split into training and validation
train_X, holdout_X, train_y, holdout_y = train_test_split(X, y, test_size=0.4, random_state=1)

logit_full = LogisticRegression(penalty="l2", C=1e42, solver='liblinear')
logit_full.fit(train_X, train_y)

print(f'Intercept {logit_full.intercept_[0]:.5f}')
with pd.option_context('display.max_columns', None):
    print(pd.DataFrame({'Coeff': logit_full.coef_[0]},
        index=X.columns).transpose().round(3))

aic_score = mlba.AIC_score(y_true=holdout_y, y_pred=logit_full.predict(holdout_X), df=len(train_X.columns) + 1)
print(f'AIC {aic_score:.3f}')


By renaming the categories of `Flight Status`, we make sure that the event of interest (delayed) is coded as the second category.

 Partial output

**Cell 11: Evaluation of the full flight delays model – confusion matrix and gains chart**  
Predictions are made on the holdout set, and a confusion matrix (accuracy, sensitivity, specificity) is printed. Then a gains chart is plotted, showing how well the model ranks delayed flights. This provides a baseline performance measure for the full model before feature reduction.

In [ ]:
logit_reg_pred = logit_full.predict_proba(holdout_X)
full_result = pd.DataFrame({'actual': holdout_y,
                            'p(0)': [p[0] for p in logit_reg_pred],
                            'p(1)': [p[1] for p in logit_reg_pred],
                            'predicted': logit_full.predict(holdout_X)})

# confusion matrix
mlba.classificationSummary(y_true=full_result['actual'], y_pred=full_result['predicted'])

mlba.gainsChart(full_result, ranking='p(1)', actual='actual', event_level='1-delayed',
                figsize=[5, 5])
plt.tight_layout()
plt.show()

**Cell 12: Reduced logistic regression model (aggregated categories)**  
A reduced model is created by aggregating categories: a binary indicator for Sunday/Monday (`Sun_Mon`), a combined carrier indicator for CO, MQ, DH, RU, and time‑of‑day bins (MORNING, NOON, AFTER2P, EVENING). The outcome is the same `Flight Status`. This reduces the number of predictors dramatically. The model is fitted without regularisation. Coefficients and AIC are printed, along with a confusion matrix. This reduced model may have similar predictive performance with fewer parameters, making it more interpretable.

In [ ]:
delays_red_df = pd.DataFrame({
    'Sun_Mon' : [1 if d in (1, 7) else 0 for d in delays_df['DAY_WEEK']],
    'Weather' : delays_df['Weather'],
    'CARRIER_CO_MQ_DH_RU' : [1 if d in ("CO", "MQ", "DH", "RU") else 0
                             for d in delays_df['CARRIER']],
    'MORNING' : [1 if d in (6, 7, 8, 9) else 0 for d in delays_df['CRS_DEP_TIME']],
    'NOON' : [1 if d in (10, 11, 12, 13) else 0 for d in delays_df['CRS_DEP_TIME']],
    'AFTER2P' : [1 if d in (14, 15, 16, 17, 18) else 0 for d in delays_df['CRS_DEP_TIME']],
    'EVENING' : [1 if d in (19, 20) else 0 for d in delays_df['CRS_DEP_TIME']],
    'Flight Status': delays_df['Flight Status'],
})

X = delays_red_df.drop(columns=['Flight Status'])
y = delays_red_df['Flight Status']

# split into training and validation
train_X, holdout_X, train_y, holdout_y = train_test_split(X, y, test_size=0.4,
                                                      random_state=1)

logit_red = LogisticRegression(penalty="l2", solver='liblinear', C=1e24)
logit_red.fit(train_X, train_y)

print(f'Intercept {logit_red.intercept_[0]:.5f}')
with pd.option_context('display.max_columns', None):
    print(pd.DataFrame({'coeff': logit_red.coef_[0]},
        index=X.columns).transpose().round(3))

aic_score = mlba.AIC_score(y_true=holdout_y, y_pred=logit_red.predict(holdout_X),
                           df=len(train_X.columns) + 1)
print(f'AIC {aic_score:.3f}')

# confusion matrix
mlba.classificationSummary(y_true=holdout_y, y_pred=logit_red.predict(holdout_X))

 Modified output

**Cell 13: Lasso (L1‑regularised) logistic regression on the full set of dummy variables**  
Using the original full set of dummy variables (from Cell 10), a logistic regression with L1 penalty is fitted using `LogisticRegressionCV` with 5‑fold cross‑validation to automatically select the regularisation strength. L1 penalty can shrink coefficients to zero, effectively performing feature selection. The resulting coefficients (many may be zero) and AIC are printed, along with a confusion matrix on the holdout set. This model may achieve similar or better performance with a sparse set of predictors.

In [ ]:
predictors = ['DAY_WEEK', 'CRS_DEP_TIME', 'ORIGIN', 'DEST', 'CARRIER', 'Weather']
outcome = 'Flight Status'

X = pd.get_dummies(delays_df[predictors], drop_first=True)
y = delays_df['Flight Status']

# split into training and validation
train_X, l1_holdout_X, train_y, l1_holdout_y = train_test_split(X, y, test_size=0.4,
                                                      random_state=1)

logit_l1 = LogisticRegressionCV(penalty="l1", solver='liblinear', cv=5)
logit_l1.fit(train_X, train_y)

print(f'Intercept {logit_l1.intercept_[0]:.5f}')
with pd.option_context('display.max_columns', None):
    print(pd.DataFrame({'coeff': logit_l1.coef_[0]}, index=X.columns).transpose().round(3))

aic_score = mlba.AIC_score(y_true=holdout_y, y_pred=logit_l1.predict(l1_holdout_X),
                           df=len(train_X.columns) + 1)
print(f'AIC {aic_score:.3f}')

# confusion matrix
mlba.classificationSummary(y_true=holdout_y, y_pred=logit_l1.predict(l1_holdout_X))

**Cell 14: Gains chart comparison of full, reduced, and Lasso models**  
The gains curves for the full model (from Cell 11), the reduced model (from Cell 12), and the Lasso model (from Cell 13) are overlaid on the same plot. This allows a visual comparison of their ranking performance. Typically, the reduced model may perform almost as well as the full model, and Lasso may perform similarly while using far fewer predictors.

In [ ]:
red_result = pd.DataFrame({'actual': holdout_y,
        'p(1)': [p[1] for p in logit_red.predict_proba(holdout_X)]})
l1_result = pd.DataFrame({'actual': holdout_y,
        'p(1)': [p[1] for p in logit_l1.predict_proba(l1_holdout_X)]})

ax = mlba.gainsChart(full_result, ranking='p(1)', actual='actual',
        event_level='1-delayed', label='Full model', color='grey', figsize=[5, 5])
mlba.gainsChart(l1_result, ranking='p(1)', actual='actual',
        event_level='1-delayed', label='Lasso model', color='red', ax=ax)
mlba.gainsChart(red_result, ranking='p(1)', actual='actual',
        event_level='1-delayed', label='Reduced model', color='black', ax=ax)
plt.tight_layout()
plt.show()

**Cell 15: Statsmodels logistic regression summary (for Universal Bank data)**  
Using the same preprocessed Universal Bank data, a constant column is added, and a logistic regression is fitted using `statsmodels.GLM` with the binomial family. The summary output includes coefficient estimates, standard errors, z‑statistics, p‑values, and confidence intervals, as well as model fit statistics (e.g., pseudo‑R², log‑likelihood, AIC). This provides a more traditional statistical inference perspective, complementing the scikit‑learn implementation.

In [ ]:
# same initial preprocessing and creating dummies

# add constant column
bank_df = sm.add_constant(bank_df, prepend=True)

y = bank_df['Personal Loan']
X = bank_df.drop(columns=['Personal Loan'])

# partition data
train_X, holdout_X, train_y, holdout_y = train_test_split(X, y, test_size=0.4, random_state=1)

# use GLM (general linear model) with the binomial family to fit a logistic regression
logit_reg = sm.GLM(train_y, train_X, family=sm.families.Binomial())
logit_result = logit_reg.fit()
logit_result.summary()